# 06 – LLM giải thích (Groq) và thí nghiệm "rule bằng code hay giao cho LLM?"

Notebook này **chỉ đọc kết quả đã lưu** trong `data/processed/llm_runs/`, sinh bởi `scripts/run_llm_experiments.py`, nên chạy lại không tốn lượt gọi API.

| Thí nghiệm | Câu hỏi | Model |
|---|---|---|
| **A. Giải thích hồ sơ** | LLM có giải thích đúng, trích dẫn đúng quy chế, không bịa không? | `openai/gpt-oss-120b` |
| **B. LLM tự xét rule** | Nếu đưa cho LLM **đúng đặc tả điều kiện** mà code dùng, LLM có cho kết quả như code không? | `openai/gpt-oss-20b`, `openai/gpt-oss-120b` |

Groq bản miễn phí giới hạn 8.000 token mỗi phút và 1.000 request mỗi ngày cho mỗi model, nên cỡ mẫu nhỏ (vài chục hồ sơ).

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from src.rules import RULE_NAMES

RUNS = ROOT / 'data' / 'processed' / 'llm_runs'
pd.set_option('display.width', 180); pd.set_option('display.max_colwidth', 120)
BLUE, ORANGE, INK, INK2, GRID = '#2a78d6', '#eb6834', '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({
    'font.family': ['Arial', 'DejaVu Sans'], 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': INK2, 'axes.labelcolor': INK2, 'axes.titlecolor': INK,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'xtick.color': INK2, 'ytick.color': INK2,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'figure.dpi': 110, 'figure.facecolor': 'white',
})
read = lambda f: [json.loads(l) for l in open(f, encoding='utf-8') if l.strip()]

## A. Giải thích hồ sơ giao dịch

Mỗi hồ sơ đi qua đủ luồng: rule + mô hình bất thường → truy xuất quy chế → LLM sinh giải thích có trích dẫn → kiểm chứng tự động. Mẫu được chọn ngẫu nhiên theo từng loại tình huống trong giai đoạn 06–08/2026.

**Các chỉ số kiểm chứng tự động:**
- **Trích dẫn hợp lệ:** mã điều nằm trong các điều đã truy xuất (được ép bằng schema) và câu trích khớp nguyên văn quy chế.
- **Độ bao phủ vi phạm:** tỷ lệ rule vi phạm trong hồ sơ được LLM nêu làm lý do (nguồn "rule", đúng mã).
- **Nhắc rule không có thật:** lý do dẫn mã rule mà hồ sơ không có vi phạm đó.
- **Nhắc dấu hiệu bất thường:** khi hồ sơ có dấu hiệu từ mô hình, LLM có nêu lý do nguồn "mo_hinh" không.

In [2]:
def metrics_table(ex):
  rows = []
  for r in ex:
    e, c = r['explanation'], r['case']
    hits = {h['rule_id'] for h in c['rule_hits']}
    refs = {x['ref'] for x in e['reasons'] if x['source'] == 'rule'}
    has_anom = bool(c['anomaly']['reasons'] or c['anomaly']['burst_day'])
    n_dropped = sum('đã loại' in w for w in r['warnings'])
    rows.append({
        'txn_id': r['txn_id'], 'Loại': r['kind'], 'Mức rủi ro (code)': r['risk']['final_level'],
        'Số rule vi phạm': len(hits),
        'Bao phủ vi phạm': len(hits & refs) / len(hits) if hits else np.nan,
        'Rule không có thật': len({x for x in refs if x.startswith('R')} - hits),
        'Nêu dấu hiệu mô hình': any(x['source'] == 'mo_hinh' for x in e['reasons']) if has_anom else np.nan,
        'Trích dẫn hợp lệ': len(e['citations']), 'Trích dẫn bị loại': n_dropped,
        'Số mục cần kiểm tra': len(e['items_to_verify']), 'Số bước xử lý': len(e['next_steps']),
        'Thời gian (s)': r['latency_s'], 'Thời gian xử lý Groq (s)': r['usage'].get('api_time_s'),
        'Token vào': r['usage']['prompt_tokens'], 'Token ra': r['usage']['completion_tokens'],
    })
  return pd.DataFrame(rows)

ex_v1 = read(RUNS / 'explain_gpt-oss-120b.jsonl')
ex = read(RUNS / 'explain_gpt-oss-120b_v2.jsonl')
A1, A = metrics_table(ex_v1), metrics_table(ex)
print(f'{len(A)} hồ sơ đã giải thích (bản v2)')
A

16 hồ sơ đã giải thích (bản v2)


,txn_id,Loại,Mức rủi ro (code),Số rule vi phạm,Bao phủ vi phạm,Rule không có thật,Nêu dấu hiệu mô hình,Trích dẫn hợp lệ,Trích dẫn bị loại,Số mục cần kiểm tra,Số bước xử lý,Thời gian (s),Thời gian xử lý Groq (s),Token vào,Token ra
0,202607-031599,Cảnh báo cao (rule),Cảnh báo cao,1,1.0,0,NaN,1,0,4,4,2.59,1.878918,1655,860
1,202606-010677,Cảnh báo cao (rule),Cảnh báo cao,1,1.0,0,True,2,0,4,5,3.20,2.805294,2311,1276
2,202607-003284,Cảnh báo cao (rule),Cảnh báo cao,1,1.0,0,NaN,2,0,4,4,11.56,2.700632,1795,1257
3,202608-008650,Cảnh báo cao (rule),Cảnh báo cao,4,1.0,0,NaN,4,0,5,5,32.60,5.764969,2536,2710
4,202608-029727,Cảnh báo cao (rule),Cảnh báo cao,2,1.0,0,True,3,0,6,5,40.31,4.224726,2179,1679
5,202608-012960,Chỉ mô hình bất thường,Cần review,0,NaN,0,True,2,0,5,5,19.72,3.946548,1889,1854
6,202608-005461,Chỉ mô hình bất thường,Cần review,0,NaN,0,False,3,0,5,6,33.04,5.444808,2425,1666
7,202607-006908,Chỉ mô hình bất thường,Cần review,1,1.0,0,True,3,0,6,5,30.04,5.239904,1958,2364
8,202608-023194,Chỉ mô hình bất thường,Cần review,0,NaN,0,True,2,0,5,5,19.17,2.879600,1824,1324
9,202606-009499,Cụm mua dồn,Cần review,0,NaN,0,True,1,1,4,3,28.31,5.535399,2136,1244


In [3]:
total_cit = A['Trích dẫn hợp lệ'].sum() + A['Trích dẫn bị loại'].sum()
summary = pd.Series({
    'Hồ sơ': len(A),
    'JSON hợp lệ theo schema': f'{len(A)}/{len(A)}',
    'Trích dẫn hợp lệ / tổng trích dẫn': f"{A['Trích dẫn hợp lệ'].sum()}/{total_cit} ({A['Trích dẫn hợp lệ'].sum() / total_cit:.0%})",
    'Hồ sơ có ít nhất 1 trích dẫn hợp lệ': f"{(A['Trích dẫn hợp lệ'] > 0).mean():.0%}",
    'Bao phủ vi phạm rule (trung bình)': f"{A['Bao phủ vi phạm'].mean():.0%}",
    'Số lần nhắc rule không có thật': int(A['Rule không có thật'].sum()),
    'Nêu dấu hiệu mô hình khi có': f"{A['Nêu dấu hiệu mô hình'].dropna().mean():.0%}",
    'Thời gian xử lý Groq trung bình (s)': round(A['Thời gian xử lý Groq (s)'].mean(), 1),
    'Thời gian thực tế trung bình, gồm chờ giới hạn tốc độ (s)': round(A['Thời gian (s)'].mean(), 1),
    'Token trung bình / hồ sơ': int((A['Token vào'] + A['Token ra']).mean()),
})
summary.to_frame('Giá trị')

,Giá trị
Hồ sơ,16
JSON hợp lệ theo schema,16/16
Trích dẫn hợp lệ / tổng trích dẫn,33/34 (97%)
Hồ sơ có ít nhất 1 trích dẫn hợp lệ,100%
Bao phủ vi phạm rule (trung bình),100%
Số lần nhắc rule không có thật,0
Nêu dấu hiệu mô hình khi có,89%
Thời gian xử lý Groq trung bình (s),3.5
"Thời gian thực tế trung bình, gồm chờ giới hạn tốc độ (s)",22.7
Token trung bình / hồ sơ,3480


In [4]:
A.groupby('Loại')[['Bao phủ vi phạm', 'Trích dẫn hợp lệ', 'Trích dẫn bị loại', 'Thời gian xử lý Groq (s)']].mean().round(2)

,Bao phủ vi phạm,Trích dẫn hợp lệ,Trích dẫn bị loại,Thời gian xử lý Groq (s)
Loại,,,,
Amazon – bình thường,1.0,1.00,0.00,2.24
Chỉ mô hình bất thường,1.0,2.50,0.00,4.38
Cảnh báo cao (rule),1.0,2.40,0.00,3.47
Cần review (rule),1.0,2.00,0.00,2.61
Cụm mua dồn,1.0,1.67,0.33,3.88


### Cải tiến từ v1 lên v2

Bản v1 phát hiện hai vấn đề:
1. LLM điền **mã điều quy chế** (D24, D13…) vào trường `ref` vốn dành cho mã rule, hoặc gắn nhầm nhãn rule cho một ý về quy chế. Bộ kiểm chứng ban đầu đếm các trường hợp này thành "rule không có thật". Đọc lại thì không có rule nào bị bịa, chỉ là ghi sai trường.
2. Câu trích dẫn bị loại vì LLM **ghép nhiều khoản bằng "..."**.

Bản v2 sửa ở **schema**, không chỉ ở prompt: `ref` chỉ được nhận các mã rule có trong hồ sơ (enum) hoặc rỗng; thêm nguồn `quy_che`; yêu cầu mỗi câu trích là một câu nguyên văn; lưu lại câu trích bị loại để kiểm tra.

In [5]:
def refs_misuse(ex):
    n = 0
    for r in ex:
        hits = {h['rule_id'] for h in r['case']['rule_hits']}
        src = r.get('raw_explanation', r['explanation'])
        n += sum(1 for x in src['reasons'] if x['source'] == 'rule' and x['ref'] not in hits)
    return n
pd.DataFrame({
    'v1': {'Trích dẫn hợp lệ': A1['Trích dẫn hợp lệ'].sum(), 'Trích dẫn bị loại': A1['Trích dẫn bị loại'].sum(),
           'Lý do gắn mã không thuộc hồ sơ': refs_misuse(ex_v1), 'Bao phủ vi phạm': round(A1['Bao phủ vi phạm'].mean(), 2)},
    'v2': {'Trích dẫn hợp lệ': A['Trích dẫn hợp lệ'].sum(), 'Trích dẫn bị loại': A['Trích dẫn bị loại'].sum(),
           'Lý do gắn mã không thuộc hồ sơ': refs_misuse(ex), 'Bao phủ vi phạm': round(A['Bao phủ vi phạm'].mean(), 2)},
})

,v1,v2
Trích dẫn hợp lệ,45.0,33.0
Trích dẫn bị loại,3.0,1.0
Lý do gắn mã không thuộc hồ sơ,5.0,0.0
Bao phủ vi phạm,1.0,1.0


In [6]:
print('Câu trích bị loại ở v2:')
for r in ex:
    for w in r['warnings']:
        if 'đã loại' in w:
            print('-', r['txn_id'], w)

Câu trích bị loại ở v2:
- 202606-009499 Câu trích D24 không khớp nguyên văn quy chế – đã loại: "Dấu hiệu bất thường gồm: ... số giao dịch trong ngày cao bất thường so với nhịp mua thường ngày; ... Dấu hiệu bất thường không phải là kết luận vi phạm."


### Ví dụ giải thích

In [7]:
def show(r):
    e, c, t = r['explanation'], r['case'], r['case']['transaction']
    lines = [f"**{r['txn_id']}** – {r['kind']} – mức rủi ro (code): **{r['risk']['final_level']}**",
             f"> {t['date']} · {t['merchant']} · {t['item_desc']} · ${t['amount']:,.2f}", '',
             f"**Tóm tắt:** {e['summary']}", '', '**Lý do:**']
    lines += [f"- [{x['source']}{' ' + x['ref'] if x['ref'] else ''}] {x['text']}" for x in e['reasons']]
    lines += ['', '**Căn cứ quy chế:**'] + [f"- {x['article_id']}: \"{x['quote']}\"" for x in e['citations']]
    lines += ['', '**Cần kiểm tra:**'] + [f'- {x}' for x in e['items_to_verify']]
    lines += ['', '**Bước tiếp theo:**'] + [f'- {x}' for x in e['next_steps']]
    if r['warnings']:
        lines += ['', '**Cảnh báo kiểm chứng:** ' + '; '.join(r['warnings'])]
    display(Markdown('\n'.join(lines)))

for kind in ['Chỉ mô hình bất thường', 'Cụm mua dồn']:
    r = next((x for x in ex if x['kind'] == kind), None)
    if r: show(r)

**202608-012960** – Chỉ mô hình bất thường – mức rủi ro (code): **Cần review**
> 19/08/2026 · TST CAJUN CORNER 1 · GENERAL PURCHASE · $325.00

**Tóm tắt:** Giao dịch được xếp mức Cần review do có dấu hiệu bất thường: số tiền cao gấp 7.2 lần mức thường, lần đầu chi cho nhóm FOOD và lần đầu mua tại nhà cung cấp TST CAJUN CORNER 1, theo quy định D24 nên kế toán cần kiểm tra chứng từ và xác nhận tính hợp lệ.

**Lý do:**
- [mo_hinh] Số tiền $325.00 gấp 7.2 lần mức chi thường gặp tại cửa hàng này.
- [mo_hinh] Lần đầu người này chi cho nhóm chi phí FOOD.
- [mo_hinh] Lần đầu người này mua ở cửa hàng TST CAJUN CORNER 1.
- [quy_che] Theo Điều 24, giao dịch có dấu hiệu bất thường được xếp mức Cần review.

**Căn cứ quy chế:**
- D24: "Dấu hiệu bất thường gồm: số tiền cao hơn nhiều lần mức thường chi của người chi, tại nhà cung cấp hoặc của đơn vị cho nhóm chi phí đó; lần đầu chi cho một nhóm chi phí hoặc một nhà cung cấp; số giao dịch trong ngày cao bất thường so với nhịp mua thường ngày; ghi sổ chậm bất thường."
- D24: "Giao dịch có vi phạm mức Trung bình hoặc có dấu hiệu bất thường so với thói quen chi tiêu được xếp Cần review."

**Cần kiểm tra:**
- Hóa đơn chi tiết từ nhà cung cấp TST CAJUN CORNER 1
- Biên nhận hoặc chứng từ mô tả chi tiết các mặt hàng mua (không chỉ "GENERAL PURCHASE")
- Xác nhận của người chi về lý do chi số tiền cao và lần đầu chi cho nhóm FOOD và nhà cung cấp này
- Kiểm tra danh sách nhà cung cấp của đơn vị xem TST CAJUN CORNER 1 có trong danh sách hay không
- Kiểm tra mức chi thường xuyên của người này cho nhóm FOOD để so sánh mức $325

**Bước tiếp theo:**
- Yêu cầu người chi cung cấp hóa đơn chi tiết hoặc bản kê mặt hàng cho giao dịch này
- Nếu nhà cung cấp chưa có trong danh sách, yêu cầu người chi nêu lý do lựa chọn và cung cấp báo giá
- Kiểm tra tính hợp lệ của chi phí ăn uống theo Điều 20 (đối tượng chi, mục đích)
- Kế toán ghi nhận kết luận sau khi xem xét đầy đủ chứng từ và phản hồi của người chi
- Nếu không đáp ứng quy định, trả lại hoặc không thanh toán khoản chi này

**202606-009499** – Cụm mua dồn – mức rủi ro (code): **Cần review**
> 25/06/2026 · INTERNATIONAL SOCIETY OF · ISA Tree Risk Assessment Q EA · $50.00

**Tóm tắt:** Giao dịch được xếp mức Cần review do có dấu hiệu bất thường về số lượng giao dịch trong ngày, theo quy chế Điều 26 và Điều 24.

**Lý do:**
- [mo_hinh] Giao dịch có dấu hiệu bất thường về số lượng giao dịch trong ngày cao bất thường so với nhịp mua thường ngày của người chi.
- [quy_che] Theo Điều 26, trường hợp người chi thực hiện số lượng giao dịch trong một ngày cao bất thường so với nhịp mua thường ngày của chính người đó, kế toán xem xét toàn bộ các giao dịch trong ngày như một hồ sơ.
- [quy_che] Theo Điều 24, dấu hiệu bất thường như số giao dịch trong ngày cao bất thường sẽ đưa giao dịch vào mức Cần review.

**Căn cứ quy chế:**
- D26: "Trường hợp người chi thực hiện số lượng giao dịch trong một ngày cao bất thường so với nhịp mua thường ngày của chính người đó, kế toán xem xét toàn bộ các giao dịch trong ngày như một hồ sơ."

**Cần kiểm tra:**
- Hóa đơn hoặc biên nhận chi tiết cho mỗi giao dịch trong ngày 25/06/2026
- Bản giải trình mục đích mua sắm và danh sách các giao dịch cùng ngày
- Bảng chi tiêu hàng ngày của người chi để so sánh nhịp mua thường
- Xác nhận của thủ trưởng đơn vị (nếu cần)

**Bước tiếp theo:**
- Kiểm tra chứng từ và bản giải trình để xác nhận mục đích mua sắm
- Đối chiếu tổng giá trị các giao dịch trong ngày với ngân sách nhóm chi phí
- Nếu không có vi phạm quy chế, phê duyệt giao dịch; nếu có dấu hiệu vi phạm, yêu cầu giải trình thêm hoặc từ chối

**Cảnh báo kiểm chứng:** Câu trích D24 không khớp nguyên văn quy chế – đã loại: "Dấu hiệu bất thường gồm: ... số giao dịch trong ngày cao bất thường so với nhịp mua thường ngày; ... Dấu hiệu bất thường không phải là kết luận vi phạm."

### Phiếu đánh giá thủ công

Kiểm chứng tự động chỉ bắt được trích dẫn sai và rule bịa, không đánh giá được **tính hữu ích**. File dưới đây dùng để người (bạn, cô, kế toán) chấm điểm 1–5 theo 3 tiêu chí.

In [8]:
out = ROOT / 'data' / 'labeling' / 'explanation_rating.xlsx'
if not out.exists():
    sheet = pd.DataFrame([{
        'txn_id': r['txn_id'], 'Loại': r['kind'], 'Mức rủi ro': r['risk']['final_level'],
        'Tóm tắt': r['explanation']['summary'],
        'Lý do': '\n'.join(f"- {x['text']}" for x in r['explanation']['reasons']),
        'Trích dẫn': '\n'.join(f"- {x['article_id']}: {x['quote']}" for x in r['explanation']['citations']),
        'Cần kiểm tra': '\n'.join(f'- {x}' for x in r['explanation']['items_to_verify']),
        'Bước tiếp theo': '\n'.join(f'- {x}' for x in r['explanation']['next_steps']),
        'Đúng với hồ sơ (1-5)': None, 'Hữu ích cho kế toán (1-5)': None, 'Rõ ràng (1-5)': None, 'Nhận xét': None,
    } for r in ex])
    sheet.to_excel(out, index=False)
    print('Đã tạo', out)
else:
    print('Đã có file chấm điểm, không ghi đè:', out)

Đã tạo D:\Thesis\data\labeling\explanation_rating.xlsx


## B. Thí nghiệm: giao việc kiểm tra rule cho LLM

**Thiết lập:** cùng một **đặc tả điều kiện** của 9 rule (nguyên văn ngưỡng trong `config/rules.toml`) được đưa cho LLM. Với mỗi giao dịch, LLM nhận đủ dữ liệu cần thiết:
- giao dịch hiện tại;
- lịch sử cùng cửa hàng trong 7 ngày;
- ngân sách tháng và số đã chi trước giao dịch;
- cửa hàng có trong danh sách đã duyệt hay không.

LLM trả về danh sách rule vi phạm, và kết quả được so với rule engine (code).

Mẫu gồm khoảng 4 giao dịch dương tính cho mỗi rule và 8 giao dịch không vi phạm, trong giai đoạn 01–08/2026. Đây là một thí nghiệm có kiểm soát: điều kiện giống hệt nhau, chỉ khác **ai thực hiện** (code hay LLM).

In [9]:
from src.data_loader import load_pcard
from src.labels import add_expense_group
from src.rules import run_rules, load_reference
_df = add_expense_group(load_pcard()); _b, _a = load_reference()
TRUTH = run_rules(_df, budgets=_b, approved=_a).groupby('txn_id').rule_id.agg(lambda s: sorted(set(s)))

def load_rc(model):
    f = RUNS / f'rulecheck_{model}.jsonl'
    if not f.exists():
        return None
    d = pd.DataFrame(read(f)).rename(columns={'truth': 'truth_at_run'})
    d['truth'] = d.txn_id.map(lambda t: TRUTH.get(t, []))      # nhãn đúng theo rule engine HIỆN TẠI
    return d

RC = {m: load_rc(m) for m in ['gpt-oss-20b', 'gpt-oss-120b']}
RC = {m: d for m, d in RC.items() if d is not None}
RULES = [f'R0{i}' for i in range(1, 10)]

def per_rule(d):
    rows = {}
    for r in RULES:
        t = d.truth.map(lambda x: r in x); p = d.pred.map(lambda x: r in x)
        tp, fp, fn = (t & p).sum(), (~t & p).sum(), (t & ~p).sum()
        rows[r] = {'Quy tắc': RULE_NAMES[r], 'Dương tính thật': int(t.sum()),
                   'Precision': tp / (tp + fp) if tp + fp else np.nan, 'Recall': tp / (tp + fn) if tp + fn else np.nan,
                   'Báo thừa': int(fp), 'Bỏ sót': int(fn)}
    return pd.DataFrame(rows).T

overall = []
for m, d in RC.items():
    exact = (d.truth.map(tuple) == d.pred.map(tuple)).mean()
    pr = per_rule(d)
    overall.append({'Cách thực hiện': f'LLM {m}', 'Số giao dịch': len(d), 'Đúng hoàn toàn (cả bộ rule)': exact,
                    'Tổng báo thừa': int(pr['Báo thừa'].sum()), 'Tổng bỏ sót': int(pr['Bỏ sót'].sum()),
                    'Thời gian TB / giao dịch (s)': d.latency_s.mean(),
                    'Token TB / giao dịch': (d.prompt_tokens + d.completion_tokens).mean()})
overall.append({'Cách thực hiện': 'Code (rule engine)', 'Số giao dịch': 485616, 'Đúng hoàn toàn (cả bộ rule)': 1.0,
                'Tổng báo thừa': 0, 'Tổng bỏ sót': 0, 'Thời gian TB / giao dịch (s)': 8 / 485616, 'Token TB / giao dịch': 0})
pd.DataFrame(overall).set_index('Cách thực hiện').style.format(
    {'Đúng hoàn toàn (cả bộ rule)': '{:.0%}', 'Thời gian TB / giao dịch (s)': '{:.5f}', 'Token TB / giao dịch': '{:,.0f}'})

,Số giao dịch,Đúng hoàn toàn (cả bộ rule),Tổng báo thừa,Tổng bỏ sót,Thời gian TB / giao dịch (s),Token TB / giao dịch
Cách thực hiện,,,,,,
LLM gpt-oss-20b,44,89%,3,2,13.98023,"2,343"
LLM gpt-oss-120b,44,89%,1,4,14.49659,"2,263"
Code (rule engine),485616,100%,0,0,0.00002,0


In [10]:
for m, d in RC.items():
    print(f'=== {m}')
    display(per_rule(d).style.format({'Precision': '{:.2f}', 'Recall': '{:.2f}'}))

=== gpt-oss-20b


,Quy tắc,Dương tính thật,Precision,Recall,Báo thừa,Bỏ sót
R01,Thiếu chứng từ / mô tả,9,1.00,1.00,0,0
R02,Vượt hạn mức phê duyệt,5,1.00,1.00,0,0
R03,Trùng hóa đơn,4,1.00,1.00,0,0
R04,Chia nhỏ giao dịch,4,1.00,1.00,0,0
R05,Vượt ngân sách,10,1.00,1.00,0,0
R06,Giao dịch tương đương tiền mặt,4,1.00,1.00,0,0
R07,Cửa hàng chưa được duyệt,11,1.00,1.00,0,0
R08,Loại chi phí không được phép,4,1.00,1.00,0,0
R09,Giao dịch ngày nghỉ,6,0.57,0.67,3,2


=== gpt-oss-120b


,Quy tắc,Dương tính thật,Precision,Recall,Báo thừa,Bỏ sót
R01,Thiếu chứng từ / mô tả,9,1.00,1.00,0,0
R02,Vượt hạn mức phê duyệt,5,1.00,1.00,0,0
R03,Trùng hóa đơn,4,1.00,0.50,0,2
R04,Chia nhỏ giao dịch,4,1.00,0.50,0,2
R05,Vượt ngân sách,10,1.00,1.00,0,0
R06,Giao dịch tương đương tiền mặt,4,1.00,1.00,0,0
R07,Cửa hàng chưa được duyệt,11,1.00,1.00,0,0
R08,Loại chi phí không được phép,4,1.00,1.00,0,0
R09,Giao dịch ngày nghỉ,6,0.86,1.00,1,0


In [11]:
m0 = list(RC)[0]
d = RC[m0]
wrong = d[d.truth.map(tuple) != d.pred.map(tuple)].copy()
wrong['Báo thừa'] = [sorted(set(p) - set(t)) for t, p in zip(wrong.truth, wrong.pred)]
wrong['Bỏ sót'] = [sorted(set(t) - set(p)) for t, p in zip(wrong.truth, wrong.pred)]
wrong['Ngày'] = wrong.payload.map(lambda x: x['giao_dich_hien_tai']['ngay'])
wrong['Số tiền'] = wrong.payload.map(lambda x: x['giao_dich_hien_tai']['so_tien'])
print(f'{m0}: {len(wrong)} giao dịch sai')
wrong[['txn_id', 'Ngày', 'Số tiền', 'truth', 'pred', 'Báo thừa', 'Bỏ sót']].head(15)

gpt-oss-20b: 5 giao dịch sai


,txn_id,Ngày,Số tiền,truth,pred,Báo thừa,Bỏ sót
4,202605-016477,2026-05-13,13238.60,"[R01, R02]","[R01, R02, R09]",[R09],[]
9,202605-020330,2026-05-13,118.94,[R03],"[R03, R09]",[R09],[]
16,202606-006855,2026-06-21,73.26,"[R05, R09]",[R05],[],[R09]
21,202607-027310,2026-07-24,20.00,[R06],"[R06, R09]",[R09],[]
35,202603-006031,2026-03-15,539.01,"[R07, R09]",[R07],[],[R09]


### Thí nghiệm phát hiện lỗi của chính rule engine

Ở lần chạy đầu, giao dịch `202605-034242` được code gắn R04 (chia nhỏ giao dịch), trong khi **cả hai LLM đều không gắn**. Kiểm tra lại thì code sai: thông báo của code là *"1 giao dịch … tổng $1,004.95 vượt hạn mức $5,000"*, tự mâu thuẫn.

**Nguyên nhân:** `groupby(nhiều khóa).rolling()` của pandas trả kết quả theo thứ tự nhóm khác thứ tự dòng đầu vào (mã nhóm đánh theo lần xuất hiện đầu tiên của từng khóa trên toàn bảng), nên việc ghép theo vị trí bị lệch. Trên toàn bộ dữ liệu, R04 đã **gắn cờ sai 1.719 giao dịch và bỏ sót 1.719 giao dịch khác**; tổng số cờ (3.065) tình cờ không đổi.

**Sửa:** tính cửa sổ thời gian bằng tìm kiếm nhị phân trên khóa (nhóm, ngày) đã sắp xếp (`src/window.py`), kèm kiểm thử tái hiện đúng "bẫy thứ tự nhóm". Nhãn đúng trong các bảng trên đã được tính lại bằng rule engine sau khi sửa.

**Bài học:** so sánh hai cách thực hiện độc lập (code và LLM) giúp lộ ra lỗi; những ca bất đồng là nơi đáng kiểm tra. Rule bằng code vẫn cần được kiểm thử kỹ, vì code sai thì sai một cách nhất quán trên mọi giao dịch.

In [12]:
changed = pd.concat([d.assign(model=m) for m, d in RC.items()])
changed = changed[changed.truth_at_run.map(tuple) != changed.truth.map(tuple)]
changed[['model', 'txn_id', 'truth_at_run', 'truth', 'pred']].rename(columns={
    'truth_at_run': 'Nhãn lúc chạy (code có lỗi)', 'truth': 'Nhãn sau khi sửa code', 'pred': 'LLM'})

,model,txn_id,Nhãn lúc chạy (code có lỗi),Nhãn sau khi sửa code,LLM
30,gpt-oss-20b,202605-034242,"[R01, R04, R05, R07, R08]","[R01, R05, R07, R08]","[R01, R05, R07, R08]"
30,gpt-oss-120b,202605-034242,"[R01, R04, R05, R07, R08]","[R01, R05, R07, R08]","[R01, R05, R07, R08]"


**Đọc kết quả:** xem nhận xét ở mục Kết luận. Các lỗi thường gặp của LLM là các phép tính mà code làm tuyệt đối chính xác: xác định thứ trong tuần hoặc ngày lễ từ một ngày cụ thể, cộng tiền trong cửa sổ thời gian (R04), so sánh ngưỡng nhiều vế của ngân sách (R05), so khoảng cách ngày (R03).

## Kết luận

In [13]:
lines = []
for m, d in RC.items():
    pr = per_rule(d)
    worst = pr.assign(err=pr['Báo thừa'] + pr['Bỏ sót']).sort_values('err', ascending=False).head(3)
    lines.append(f"- **LLM {m}:** đúng hoàn toàn {(d.truth.map(tuple) == d.pred.map(tuple)).mean():.0%} số giao dịch; "
                 f"sai nhiều nhất ở {', '.join(f'{i} ({int(r.err)} lỗi)' for i, r in worst.iterrows() if r.err > 0)}; "
                 f"{d.latency_s.mean():.1f} giây và {int((d.prompt_tokens + d.completion_tokens).mean()):,} token mỗi giao dịch.")
lines.append(f"- **Code:** đúng 100% theo định nghĩa, chạy 485.616 giao dịch trong khoảng 8 giây, không tốn token.")
lines.append(f"- **Giải thích (gpt-oss-120b):** {summary['Trích dẫn hợp lệ / tổng trích dẫn']} trích dẫn hợp lệ; bao phủ vi phạm "
             f"{summary['Bao phủ vi phạm rule (trung bình)']}; {summary['Số lần nhắc rule không có thật']} lần nhắc rule không có thật.")
display(Markdown('\n'.join(lines)))

- **LLM gpt-oss-20b:** đúng hoàn toàn 89% số giao dịch; sai nhiều nhất ở R09 (5 lỗi); 14.0 giây và 2,343 token mỗi giao dịch.
- **LLM gpt-oss-120b:** đúng hoàn toàn 89% số giao dịch; sai nhiều nhất ở R03 (2 lỗi), R04 (2 lỗi), R09 (1 lỗi); 14.5 giây và 2,262 token mỗi giao dịch.
- **Code:** đúng 100% theo định nghĩa, chạy 485.616 giao dịch trong khoảng 8 giây, không tốn token.
- **Giải thích (gpt-oss-120b):** 33/34 (97%) trích dẫn hợp lệ; bao phủ vi phạm 100%; 0 lần nhắc rule không có thật.

1. **Điều kiện xác định được bằng rule phải do code xử lý.** Ngay cả khi được cho đúng đặc tả và đủ dữ liệu, LLM vẫn sai ở các phép tính ngày, tổng tiền và ngưỡng; kết quả còn không ổn định, chậm hơn và tốn chi phí. Kết quả này là căn cứ thực nghiệm cho yêu cầu của đề tài.
2. **LLM phù hợp với việc diễn giải:** khi nhận hồ sơ đã tính sẵn kèm điều quy chế, LLM giải thích đầy đủ các vi phạm, trích dẫn đúng nguyên văn và đề xuất việc cần kiểm tra. Mức rủi ro vẫn do code quyết định.
3. **Kiểm chứng tự động** (schema giới hạn mã điều, đối chiếu nguyên văn câu trích, phát hiện rule không có thật) là lớp bảo vệ cần thiết. Phần đánh giá chất lượng còn lại cần người chấm (`data/labeling/explanation_rating.xlsx`).
4. **So sánh độc lập giúp kiểm tra chính code:** thí nghiệm này đã làm lộ một lỗi trong rule R04. Vì vậy rule bằng code phải đi kèm kiểm thử, còn LLM có thể dùng như một bộ kiểm tra chéo ngoại tuyến (offline), không dùng để quyết định.
5. **Hạn chế:** cỡ mẫu nhỏ do giới hạn bản miễn phí của Groq; tập đặc tả và dữ liệu do người xây dựng hệ thống chuẩn bị.